# Top Coffee Dataset Initial Inspection/Cleaning

## Imports/Pathways

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import os

In [2]:
# File Pathway
path = r"C:\Users\Chase\anaconda_projects\Exercise_6_Coffee\A6_Coffee"

In [3]:
# Import dataset
df_coffee = pd.read_csv(
    r'C:\Users\Chase\anaconda_projects\Exercise_6_Coffee\A6_Coffee\02_Data\Prepared_Data\Top Rated Coffee\top-rated-coffee-edited.csv')

## Data Inspection
### Table of Contents
1. Overview
2. Missing Values
3. Duplicate Check
4. Spelling & Encoding Issues
5. Price and Measurement
6. Roast Level

### 1. Overview

In [4]:
# General check of first 10 rows
df_coffee.head()

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,price_per_ounce,origin_country
0,Colombia Finca Campo Hermosa,94.0,"Cleveland, Tennessee","Quindio Department, Colombia",Light,$29.99,8 ounces,82.0,29.990000,Colombia
1,Colombia Finca La Sirena Mango Co-Ferment,94.0,"Cleveland, Tennessee","Quindio Department, Colombia",Light,$22.99,8 ounces,87.0,22.990000,Colombia
2,In Bloom,94.0,"Jersey City, New Jersey",Colombia; Ethiopia,Light,$25.00,250 grams,88.0,0.881834,Ethiopia
3,Ethiopia Washed Kaffa Gimbo Lot Rich Espresso,96.0,"Chia-Yi, Taiwan","Gimbo, Kaffa Province, Ethiopia",Medium Light,NT $250,8 ounces,77.0,250.000000,Ethiopia
4,Ethiopia Natural Gute Bona,95.0,"Chia-Yi, Taiwan","Sidamo growing region, southern Ethiopia",Medium Light,NT $400,8 ounces,78.0,400.000000,Ethiopia


In [5]:
# Check the dataset shape
df_coffee.shape

(2215, 10)

In [6]:
# List all column headers
df_coffee.columns.tolist()

['coffee_name',
 'total_score',
 'roaster_location',
 'coffee_origin',
 'roast_level',
 'price_amount',
 'price_unit',
 'agtron_roast',
 'price_per_ounce',
 'origin_country']

In [7]:
# Check data types
df_coffee.dtypes

coffee_name          object
total_score         float64
roaster_location     object
coffee_origin        object
roast_level          object
price_amount         object
price_unit           object
agtron_roast        float64
price_per_ounce     float64
origin_country       object
dtype: object

In [8]:
# Descriptive statistics
df_coffee.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
coffee_name,2215,2027,Sumatra Tano Batak,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_score,2212.0,NaN,NaN,NaN,94.536618,0.758499,94.0,94.0,94.0,95.0,98.0
roaster_location,2215,255,"Madison, Wisconsin",214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
coffee_origin,2209,832,"Boquete growing region, western Panama",102,NaN,NaN,NaN,NaN,NaN,NaN,NaN
roast_level,2192,6,Medium Light,1263,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_amount,2101,481,$20.00,67,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_unit,2096,90,12 ounces,839,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agtron_roast,2205.0,NaN,NaN,NaN,76.576417,16.54319,0.0,74.0,78.0,81.0,689.0
price_per_ounce,1956.0,NaN,NaN,NaN,110.145506,271.820033,0.282187,18.0,23.0,47.3125,5800.0
origin_country,1774,32,Ethiopia,545,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2. Missing Values

In [9]:
# Create a summary table of missing values
missing_count = df_coffee.isnull().sum()
missing_percent = (df_coffee.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_percent
})

In [10]:
# Filter to show only columns with missing values
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

missing_summary

,Missing Count,Missing %
origin_country,441,19.91
price_per_ounce,259,11.69
price_unit,119,5.37
price_amount,114,5.15
roast_level,23,1.04
agtron_roast,10,0.45
coffee_origin,6,0.27
total_score,3,0.14


A wide range of missing values but percentages are within reason

### 3. Duplicate Check

In [11]:
# Check for duplicate rows
df_coffee.duplicated().sum()

np.int64(0)

No duplicated rows found.

### 4. Spelling & Encoding Issues

With the assistance of Copilot and explaination I was able to determine all the text that are spelt odd.

In [12]:
# Step 1: Define a function to check for non-ASCII characters
def has_non_ascii(text):
    return any(ord(char) > 127 for char in str(text))
    
# Convert the input to string (in case it's not already)
# Loop through each character in the string
# If any character has a Unicode value > 127, it's non-ASCII (e.g. Ã, ñ, â€™)

In [13]:
# Step 2: Apply the function to key text columns to flag rows with weird characters
for col in ['coffee_name', 'roaster_location', 'coffee_origin', 'origin_country']:
    df_coffee[f'{col}_non_ascii'] = df_coffee[col].apply(has_non_ascii)
    
# Create a new column like 'coffee_name_non_ascii' with True/False values
# True means that row has non-ASCII characters in that column

In [14]:
# Step 3: Filter rows with any non-ASCII flags
df_encoding_issues = df_coffee[
    (df_coffee['coffee_name_non_ascii']) |
    (df_coffee['roaster_location_non_ascii']) |
    (df_coffee['coffee_origin_non_ascii']) |
    (df_coffee['origin_country_non_ascii'])
]

In [15]:
# Step 4: Set up a preview to see all flagged rows
# Configure pandas display settings for full visibility
pd.set_option('display.max_rows', None)           # Show all rows
pd.set_option('display.max_columns', None)        # Show all columns
pd.set_option('display.max_colwidth', None)       # Show full text in each cell
pd.set_option('display.expand_frame_repr', False) # Prevent line wrapping

# Original filtering logic
flagged = df_encoding_issues[
    df_encoding_issues['origin_country'].isna() | 
    (df_encoding_issues['origin_country'].str.len() < 4)
]

# Preview flagged rows with full visibility
flagged[['coffee_name', 'roaster_location', 'coffee_origin', 'origin_country']]

,coffee_name,roaster_location,coffee_origin,origin_country
26,Kaʻū Morning Glory,"Hilo, Hawai’i Island, Hawai’i","Kaʻū growing region, Hawai’i Island, Hawai’i",NaN
27,Kaʻū Yellow Bourbon,"Hilo, Hawai’i Island, Hawai’i","Kaʻū growing region, Hawai’i Island, Hawai’i",NaN
34,Ka’u Giant Maragogipe,"Hilo, Hawai'i","Ka’u growing region, Hawai’i Island, Hawai’i",NaN
35,Ka’u Red Bourbon,"Hilo, Hawai'i","Ka’u growing region, Hawai’i Island, Hawai’i",NaN
36,Kona Bloom,"Hilo, Hawai'i","Kona District, Hawai’i Island, Hawai’i",NaN
64,Kona Tropical Punch Washed,"Hilo, Hawai’i Island, Hawai’i","Kona growing region, Hawai’i Island, Hawai'i",NaN
92,Kona Geisha Champagne Natural,"Hilo, Hawai’i Island, Hawai’i","Hōlualoa, North Kona growing region, Hawai’i Island, Hawai’i",NaN
107,Wai Meli Morning,"Mountain View, Hawai'i","Kona and Ka’u, Hawai’i; other undisclosed origins",NaN
149,Kahiko Orange,"Holualoa, Hawai’i","Holualoa, North Kona growing district, “Big Island” of Hawai’i",NaN
151,Kona Orange,"Holualoa, Hawai’i","Holualoa, North Kona growing district, “Big Island” of Hawai’i",NaN


### 5. Price and Measurement

I want to make price/weigth to be consistant and here I'll identify what I have to work with.

In [16]:
# Preview unique price formats — check for currency and unit inconsistencies
df_coffee['price_unit'].unique()

array(['8 ounces', '250 grams', '4 ounces', '12 ounces', '200 grams',
       '100 grams', '10 ounces', ' 12 ounces', '227 grams', '170 grams',
       '113 grams', '50 grams', '20 grams', '6 ounces', '120 grams', nan,
       '125 grams', '225 grams', '.8 ounces', '220 grams', '7 ounces',
       '150 ml bottle', '150 grams', '10 ounces (sold as a set of 3',
       '455 grams', '454 grams', '230 grams', '12ounces',
       '4 ounces (sold only as a two-pack)', '240 grams', '310 grams',
       '175 grams', '200g', '14 ounces', '16 ounces', '60 grams',
       '10 capsules', '5.5 ounces', '105 grams', ' 227 grams',
       ' 8 ounces', '227g', '10.5 ounces', '150-gram packet',
       '70 grams; $200.00', '300 grams', '5 ounces', '160-ml bottle',
       '12-ounce bottle', '150-gram tin', '375 ml. flask', '115 grams',
       '8-ounce can', 'can', '50 ounces', '150 ml. bottle',
       '8 ounces (currently on sale for $36.76)', '12 ounces ($79.00',
       ' 7 ounces', '18 grams', '4 ounces; limite

With this information I inputted into Copilot to id all the unquie characters.

In [17]:
# Extract currency symbols or codes using regex
df_coffee['price_currency'] = df_coffee['price_amount'].str.extract(
    r'(NTD\$|NT\$|USD\$|US\$|CAD\$|AUD\$|HKD\$|HK\$|KRW\$|KRW|RMB\$|CNY\$|IDR\$|¥|£|€|\$)',
    expand=False
)

In [18]:
# Count unique currency formats
currency_counts = df_coffee['price_currency'].value_counts(dropna=False)
currency_counts

price_currency
$       2030
NaN      132
NT$       31
KRW       10
£          6
¥          4
NTD$       1
KRW$       1
Name: count, dtype: int64

In [19]:
# Measurement Units
# Extract unit keyword from each string
df_coffee['unit_keyword'] = (
    df_coffee['price_unit']
    .astype(str)
    .str.lower()
    .str.extract(r'(gram|g|kg|ounce|oz|lb|pound|ml|fl oz|fluid ounce|capsule|can|bottle)')
)

In [20]:
# Count frequency of each unit keyword
unit_keyword_counts = df_coffee['unit_keyword'].value_counts(dropna=False)
unit_keyword_counts

unit_keyword
ounce          1726
gram            333
NaN             120
oz               17
ml                6
g                 5
capsule           5
can               1
fluid ounce       1
kg                1
Name: count, dtype: int64

In [21]:
# Count unique units formats
unit_counts = df_coffee['price_unit'].value_counts(dropna=False)
unit_counts

price_unit
12 ounces                                  839
8 ounces                                   522
4 ounces                                   165
NaN                                        119
16 ounces                                   88
227 grams                                   71
200 grams                                   68
100 grams                                   50
250 grams                                   37
6 ounces                                    36
10 ounces                                   36
225 grams                                   24
150 grams                                   13
230 grams                                    9
12 oz.                                       8
20 grams                                     7
125 grams                                    6
7 ounces                                     6
454 grams                                    5
 12 ounces                                   5
10 capsules                                  5
22

### 6. Roast Level

In [22]:
# Preview roast level categories — check for casing and punctuation variants
df_coffee['roast_level'].value_counts()

roast_level
Medium Light    1263
Light            630
Medium           244
Medium Dark       49
Dark               5
Very Dark          1
Name: count, dtype: int64

The format for roast_levels are fine

## Data Cleaning
### Table of Contents
1. Spelling and Encoding Issues 
2. Price and Measurement Unit Format Inconsistencies
3. Dropping Columns
4. Dropping Rows with Missing Vital Data
5. Missing values
6. Export

### 1. Spelling and Encoding Issues

In [23]:
# Create a clean copy of rows with missing or suspicious origin_country values
df_encoding_issues = df_coffee[df_coffee['origin_country'].isna() | (df_coffee['origin_country'].str.len() < 4)].copy()

In [24]:
import re

def normalize_encoding(text):
    if pd.isna(text):
        return text
    # Remove all apostrophe-like characters and quotes
    text = re.sub(r"[ʻ’‘'\"“”]", "", text)
    # Replace dashes and ellipses
    text = text.replace("–", "-").replace("…", "...").strip()
    return text

In [25]:
# Apply normalization to all relevant columns
for col in ['coffee_name', 'roaster_location', 'coffee_origin', 'origin_country']:
    df_encoding_issues[col] = df_encoding_issues[col].apply(normalize_encoding)

In [26]:
# Assign 'United States' for Hawaii-related origins
hawaii_keywords_raw = [
    'hawaii', 'hawai’i', 'big island', 'kona', 'ka’u', 'puna',
    'holualoa', 'mountain view', 'captain cook', 'kurtistown', 'kainaliu'
]

hawaii_keywords = [normalize_encoding(k).lower() for k in hawaii_keywords_raw]


def infer_hawaii_country(origin, current_country):
    if pd.notna(current_country):
        return current_country
    if pd.isna(origin):
        return current_country
    origin_lower = origin.lower()
    if any(keyword in origin_lower for keyword in hawaii_keywords):
        return 'United States'
    return current_country

In [27]:
# Extract known country names from origin string
known_countries = [
    'panama', 'guatemala', 'ethiopia', 'kenya', 'nicaragua', 'ecuador', 'costa rica'
]

def extract_country_from_origin(origin, current_country):
    if pd.notna(current_country):
        return current_country
    if pd.isna(origin):
        return current_country
    origin_lower = origin.lower()
    for country in known_countries:
        if country in origin_lower:
            return country.title()
    return current_country

In [28]:
# Infer 'United States' from roaster location if origin is missing
def infer_from_roaster_location(roaster, current_country):
    if pd.notna(current_country):
        return current_country
    if pd.isna(roaster):
        return current_country
    if 'hawaii' in roaster.lower() or 'hawai’i' in roaster.lower():
        return 'United States'
    return current_country

In [29]:
# Apply all inference functions in order of priority
df_encoding_issues['origin_country'] = df_encoding_issues.apply(
    lambda row: infer_hawaii_country(
        row['coffee_origin'],
        extract_country_from_origin(
            row['coffee_origin'],
            infer_from_roaster_location(row['roaster_location'], row['origin_country'])
        )
    ),
    axis=1
)

In [30]:
# Update original DataFrame with cleaned origin_country values
df_coffee.loc[df_encoding_issues.index, 'origin_country'] = df_encoding_issues['origin_country']

In [31]:
# Work check and preview cleaned rows by index
df_coffee.loc[df_encoding_issues.index, ['coffee_name', 'roaster_location', 'coffee_origin', 'origin_country']]

,coffee_name,roaster_location,coffee_origin,origin_country
26,Kaʻū Morning Glory,"Hilo, Hawai’i Island, Hawai’i","Kaʻū growing region, Hawai’i Island, Hawai’i",United States
27,Kaʻū Yellow Bourbon,"Hilo, Hawai’i Island, Hawai’i","Kaʻū growing region, Hawai’i Island, Hawai’i",United States
34,Ka’u Giant Maragogipe,"Hilo, Hawai'i","Ka’u growing region, Hawai’i Island, Hawai’i",United States
35,Ka’u Red Bourbon,"Hilo, Hawai'i","Ka’u growing region, Hawai’i Island, Hawai’i",United States
36,Kona Bloom,"Hilo, Hawai'i","Kona District, Hawai’i Island, Hawai’i",United States
64,Kona Tropical Punch Washed,"Hilo, Hawai’i Island, Hawai’i","Kona growing region, Hawai’i Island, Hawai'i",United States
92,Kona Geisha Champagne Natural,"Hilo, Hawai’i Island, Hawai’i","Hōlualoa, North Kona growing region, Hawai’i Island, Hawai’i",United States
106,Pueo Blend,"Hilo, Hawai'i",Guatemala and other undisclosed origins,United States
107,Wai Meli Morning,"Mountain View, Hawai'i","Kona and Ka’u, Hawai’i; other undisclosed origins",United States
149,Kahiko Orange,"Holualoa, Hawai’i","Holualoa, North Kona growing district, “Big Island” of Hawai’i",United States


The remaining 84 rows are due to missing data which will be handed in the missing data section.

### 2. Price and Measurement Unit Format Inconsistencies

#### Price Inconsistencies

In [32]:
# Normalize and clean the various price units
def clean_currency(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    mapping = {
        '$': 'USD',
        'usd $': 'USD',
        'us $': 'USD',
        'cad $': 'CAD',
        'aud $': 'AUD',
        'nt$': 'TWD',
        'ntd$': 'TWD',
        'nt $': 'TWD',
        'hkd $': 'HKD',
        'hk $': 'HKD',
        'krw': 'KRW',
        'krw $': 'KRW',
        'rmb $': 'CNY',
        'cny $': 'CNY',
        '¥': 'JPY',
        '£': 'GBP',
        'idr $': 'IDR'
    }
    return mapping.get(value, value.upper())

In [33]:
# Create a new column, price_currency_clean with normalized currencies.
df_coffee['price_currency_clean'] = df_coffee['price_currency'].apply(clean_currency)

In [34]:
# Exchange rates table to USD
exchange_rates = {
    'USD': 1.0,
    'TWD': 0.03268,
    'KRW': 0.0007042,
    'GBP': 1.3407,
    'JPY': 0.006638,
    'AUD': 0.6518,
    'CAD': 0.7125,
    'CNY': 0.1404,
    'IDR': 0.00006031,
    'HKD': 0.1287
}

In [35]:
# Step 1: Extract Currency Symbol or Lab
# Pulls the currency prefix from each string
df_coffee['price_currency_raw'] = (
    df_coffee['price_amount']
    .astype(str)
    .str.extract(r'^([^\d\s]+(?:\s*\$)?)')  # Captures things like $, NT $, €, ¥, etc.
)

In [36]:
# Step 2: Normalize Currency Labels
# Uses existing clean_currency() function to map the messy labels
df_coffee['price_currency_clean'] = df_coffee['price_currency_raw'].apply(clean_currency)

In [37]:
# Extract Numeric Price
df_coffee['price_amount_clean'] = (
    df_coffee['price_amount']
    .astype(str)
    .str.extract(r'(\d+\.?\d*)')
    .astype(float)
)

In [38]:
# Currency conversation
df_coffee['price_usd'] = df_coffee.apply(
    lambda row: row['price_amount_clean'] * exchange_rates.get(row['price_currency_clean'], np.nan),
    axis=1
)

In [39]:
# price_usd had more than 2 decimals so I fixed this
df_coffee['price_usd'] = df_coffee['price_usd'].round(2)

In [40]:
# Work check
df_coffee[['price_amount', 'price_currency_raw', 'price_currency_clean', 'price_amount_clean', 'price_usd']].head(10)

,price_amount,price_currency_raw,price_currency_clean,price_amount_clean,price_usd
0,$29.99,$,USD,29.99,29.99
1,$22.99,$,USD,22.99,22.99
2,$25.00,$,USD,25.00,25.00
3,NT $250,NT $,TWD,250.00,8.17
4,NT $400,NT $,TWD,400.00,13.07
5,$39.00,$,USD,39.00,39.00
6,NT $800,NT $,TWD,800.00,26.14
7,NT $350,NT $,TWD,350.00,11.44
8,NT $300,NT $,TWD,300.00,9.80
9,NT $600,NT $,TWD,600.00,19.61


#### Measurement  Inconsistencies

In [41]:
# Cean and Standardize Unit Labels
def clean_unit(value):
    if pd.isna(value):
        return np.nan
    value = str(value).lower().strip()
    if 'gram' in value or value == 'g':
        return 'g'
    elif 'kg' in value:
        return 'kg'
    elif 'ounce' in value or value == 'oz':
        return 'oz'
    elif 'lb' in value or 'pound' in value:
        return 'lb'
    else:
        return np.nan

df_coffee['price_unit_clean'] = df_coffee['price_unit'].apply(clean_unit)

In [42]:
unit_conversion_to_g = {
    'g': 1,
    'kg': 1000,
    'oz': 28.3495,
    'lb': 453.592
}

In [43]:
# Extract Numeric Quantity
df_coffee['unit_quantity'] = (
    df_coffee['price_unit']
    .astype(str)
    .str.extract(r'(\d+\.?\d*)')[0]
    .astype(float)
)

In [44]:
# Convert All to Grams
def convert_to_g(row):
    unit = row['price_unit_clean']
    qty = row['unit_quantity']
    factor = unit_conversion_to_g.get(unit)
    return qty * factor if factor and qty else np.nan

df_coffee['quantity_g'] = df_coffee.apply(convert_to_g, axis=1)

In [45]:
# Keep it to two decimals
df_coffee['quantity_g'] = df_coffee['quantity_g'].round(2)

In [46]:
# Work check
df_coffee[['price_usd', 'price_unit', 'price_unit_clean', 'unit_quantity', 'quantity_g']].head(10)

,price_usd,price_unit,price_unit_clean,unit_quantity,quantity_g
0,29.99,8 ounces,oz,8.0,226.8
1,22.99,8 ounces,oz,8.0,226.8
2,25.00,250 grams,g,250.0,250.0
3,8.17,8 ounces,oz,8.0,226.8
4,13.07,8 ounces,oz,8.0,226.8
5,39.00,8 ounces,oz,8.0,226.8
6,26.14,4 ounces,oz,4.0,113.4
7,11.44,8 ounces,oz,8.0,226.8
8,9.80,4 ounces,oz,4.0,113.4
9,19.61,4 ounces,oz,4.0,113.4


In [47]:
# Create a column combining the price (USD) with measurement (gram)
df_coffee['usd_per_gram'] = (
    '$' + df_coffee['price_usd'].round(2).astype(str)
    + ' / ' + df_coffee['quantity_g'].round(2).astype(str)
    + 'g'
)

In [48]:
# Work check
df_coffee[['price_usd', 'quantity_g', 'usd_per_gram']].head(10)

,price_usd,quantity_g,usd_per_gram
0,29.99,226.8,$29.99 / 226.8g
1,22.99,226.8,$22.99 / 226.8g
2,25.00,250.0,$25.0 / 250.0g
3,8.17,226.8,$8.17 / 226.8g
4,13.07,226.8,$13.07 / 226.8g
5,39.00,226.8,$39.0 / 226.8g
6,26.14,113.4,$26.14 / 113.4g
7,11.44,226.8,$11.44 / 226.8g
8,9.80,113.4,$9.8 / 113.4g
9,19.61,113.4,$19.61 / 113.4g


### 3. Dropping Columns

In [49]:
# Dropping price_per_unit and origin_country
df_coffee.drop(columns=['price_per_ounce', 'origin_country'], inplace=True)

The columns price_per_ounce and origin_country were dropped to streamline the dataset. price_per_ounce was removed because a more robust column was created that standardizes both currency and measurement units. origin_country was dropped due to incomplete coverage and will be reconstructed from coffee_origin using a structured extraction pipeline in a later step.

In [50]:
# Work check
print(df_coffee.columns.tolist())

['coffee_name', 'total_score', 'roaster_location', 'coffee_origin', 'roast_level', 'price_amount', 'price_unit', 'agtron_roast', 'coffee_name_non_ascii', 'roaster_location_non_ascii', 'coffee_origin_non_ascii', 'origin_country_non_ascii', 'price_currency', 'unit_keyword', 'price_currency_clean', 'price_currency_raw', 'price_amount_clean', 'price_usd', 'price_unit_clean', 'unit_quantity', 'quantity_g', 'usd_per_gram']


Both price_per_unit and origin_country were dropped.

### 4. Dropping Rows with Missing Vital Data

In [51]:
df_coffee.isna().sum().sort_values(ascending=False)

quantity_g                    154
price_unit_clean              154
price_usd                     146
price_currency                132
unit_keyword                  120
unit_quantity                 120
price_unit                    119
price_amount_clean            116
price_amount                  114
roast_level                    23
agtron_roast                   10
coffee_origin                   6
total_score                     3
coffee_name                     0
roaster_location                0
origin_country_non_ascii        0
coffee_name_non_ascii           0
roaster_location_non_ascii      0
coffee_origin_non_ascii         0
price_currency_raw              0
price_currency_clean            0
usd_per_gram                    0
dtype: int64

From this list I will go through each column to see if there are rows to be dropped

#### total_score

In [52]:
# Seeing if rows can be dropped due to missing critical fields
df_coffee[df_coffee['total_score'].isna()]

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,coffee_name_non_ascii,roaster_location_non_ascii,coffee_origin_non_ascii,origin_country_non_ascii,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram
2212,Elm City House Blend,NaN,"New Haven, Connecticut",NaN,Dark,NaN,NaN,43.0,False,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang
2213,Vienna Roast,NaN,"Lake Tahoe, California",NaN,Medium Dark,NaN,NaN,44.0,False,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang
2214,Natural Moka – Green,NaN,"Maui, Hawaii",NaN,NaN,NaN,NaN,NaN,True,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang


Retaining: None. 
Dropping: 2212-2214. missing critical fields; entry lacks usable data and appears to be a shell row.

In [53]:
# Drop rows 2212–2214
df_coffee = df_coffee[df_coffee['total_score'].notna()]

In [54]:
# Work check on the dropped rows
df_coffee[df_coffee['total_score'].isna()]

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,coffee_name_non_ascii,roaster_location_non_ascii,coffee_origin_non_ascii,origin_country_non_ascii,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram


Successfully dropped: rows 2212-2214

#### coffee_origins

In [55]:
# Updated missing value count_v1
df_coffee.isna().sum()[df_coffee.isna().sum() > 0]

coffee_origin           3
roast_level            22
price_amount          111
price_unit            116
agtron_roast            9
price_currency        129
unit_keyword          117
price_amount_clean    113
price_usd             143
price_unit_clean      151
unit_quantity         117
quantity_g            151
dtype: int64

There are only 3 of 6 coffee_origin rows missing data after the last rows were dropped. Note to self - repeat missing value check after rows are dropped

In [56]:
# Seeing if rows can be dropped due to missing critical fields
df_coffee[df_coffee['coffee_origin'].isna()]

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,coffee_name_non_ascii,roaster_location_non_ascii,coffee_origin_non_ascii,origin_country_non_ascii,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram
246,Colombia Los Nogales Yellow Bourbon,94.0,"Osaka, Japan",NaN,Light,$28.00,150 grams,86.0,False,False,False,False,$,gram,USD,$,28.0,28.0,g,150.0,150.0,$28.0 / 150.0g
2210,Finca La Tacita,95.0,"Boise, Idaho",NaN,NaN,NaN,NaN,NaN,False,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang
2211,Ethiopia Yirgacheffe,94.0,"Fairbanks, Alaska",NaN,NaN,NaN,NaN,NaN,False,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang


Retaining: row 246 - sufficient data across key fields for analysis and conversion.
Dropping: row(s) 2210, 2211 - missing critical fields; entry lacks usable data and appears to be a shell row.

In [57]:
# Drop rows 2210 & 2211
df_coffee = df_coffee.drop([2210, 2211])

In [58]:
# Work check on the dropped rows
df_coffee[df_coffee['coffee_origin'].isna()]

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,coffee_name_non_ascii,roaster_location_non_ascii,coffee_origin_non_ascii,origin_country_non_ascii,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram
246,Colombia Los Nogales Yellow Bourbon,94.0,"Osaka, Japan",NaN,Light,$28.00,150 grams,86.0,False,False,False,False,$,gram,USD,$,28.0,28.0,g,150.0,150.0,$28.0 / 150.0g


Successfully dropped: rows 2210 & 2211
Retained: row 246

In [59]:
# Updated missing value count_v2
df_coffee.isna().sum()[df_coffee.isna().sum() > 0]

coffee_origin           1
roast_level            20
price_amount          109
price_unit            114
agtron_roast            7
price_currency        127
unit_keyword          115
price_amount_clean    111
price_usd             141
price_unit_clean      149
unit_quantity         115
quantity_g            149
dtype: int64

#### roast_level

In [60]:
# Seeing if rows can be dropped due to missing critical fields
df_coffee[df_coffee['roast_level'].isna()]

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,coffee_name_non_ascii,roaster_location_non_ascii,coffee_origin_non_ascii,origin_country_non_ascii,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram
243,Brazil Ipanema Premier Cru Cherry Cold Drip,94.0,"Yilan, Taiwan","Southern Minas Gerais, Brazil",NaN,NT $200,150 ml bottle,0.0,False,False,False,False,$,ml,TWD,NT $,200.00,6.54,NaN,150.0,NaN,$6.54 / nang
277,Euphora Plumeria Cold Brew,94.0,"Taipei, Taiwan",Costa Rica,NaN,NT 200,150 ml bottle,NaN,False,False,False,False,NaN,ml,NT,NT,200.00,NaN,NaN,150.0,NaN,$nan / nang
278,Colombia El Paraiso Lychee Rose Cold Drip,94.0,"Yilan, Taiwan","Piendamó, Cauca Department, Colombia",NaN,NT 240,150 ml bottle,NaN,False,False,True,False,NaN,ml,NT,NT,240.00,NaN,NaN,150.0,NaN,$nan / nang
738,Esmeralda Estate Panama Geisha,94.0,"London, England","Boquete growing region, western Panama",NaN,£50,10 capsules,0.0,False,False,False,False,£,capsule,GBP,£,50.00,67.04,NaN,10.0,NaN,$67.04 / nang
941,Yemen Lot 106,96.0,"San Jose, California","Al Hayma region, Sana’a Governorate, Yemen",NaN,$45.00,4 ounces,0.0,False,False,True,False,$,ounce,USD,$,45.00,45.00,oz,4.0,113.40,$45.0 / 113.4g
943,Yemen Al Wadi,94.0,"San Jose, California","Al Wadi region, Sana’a Governorate, Yemen",NaN,$32.00,4 ounces,0.0,False,False,True,False,$,ounce,USD,$,32.00,32.00,oz,4.0,113.40,$32.0 / 113.4g
997,Brazil Ipanema Golden Edition C26 Lychee,94.0,"Yilan, Taiwan","Ipanema, Brazil",NaN,NT $200,160-ml bottle,0.0,False,False,False,False,$,ml,TWD,NT $,200.00,6.54,NaN,160.0,NaN,$6.54 / nang
998,Ethiopia Mormora Cold Brew,94.0,"Spokane, Washington","Guji Zone, Oromia Region, Ethiopia",NaN,$5.50,12-ounce bottle,0.0,False,False,False,False,$,ounce,USD,$,5.50,5.50,oz,12.0,340.19,$5.5 / 340.19g
1044,Bourbon Barrel Aged Ethiopia Cold Brew,94.0,"Glendale, California","Yirgacheffe growing region, southern Ethiopia",NaN,$11.00,375 ml. flask,0.0,False,False,False,False,$,ml,USD,$,11.00,11.00,NaN,375.0,NaN,$11.0 / nang
1157,Esmeralda Estate Panama Geisha,94.0,"London, England","Boquete growing region, western Panama",NaN,£45,10 capsules,0.0,False,False,False,False,£,capsule,GBP,£,45.00,60.33,NaN,10.0,NaN,$60.33 / nang


Retained: All rows currently selected. Dropped: None

#### agtron_roast

In [61]:
# Seeing if rows can be dropped due to missing critical fields
df_coffee[df_coffee['agtron_roast'].isna()]

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,coffee_name_non_ascii,roaster_location_non_ascii,coffee_origin_non_ascii,origin_country_non_ascii,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram
277,Euphora Plumeria Cold Brew,94.0,"Taipei, Taiwan",Costa Rica,NaN,NT 200,150 ml bottle,NaN,False,False,False,False,NaN,ml,NT,NT,200.00,NaN,NaN,150.0,NaN,$nan / nang
278,Colombia El Paraiso Lychee Rose Cold Drip,94.0,"Yilan, Taiwan","Piendamó, Cauca Department, Colombia",NaN,NT 240,150 ml bottle,NaN,False,False,True,False,NaN,ml,NT,NT,240.00,NaN,NaN,150.0,NaN,$nan / nang
1472,Esmeralda Estate Panama Geisha,94.0,"London, England","Boquete growing region, western Panama",NaN,£50,10 capsules,NaN,False,False,False,False,£,capsule,GBP,£,50.00,67.04,NaN,10.0,NaN,$67.04 / nang
1474,Red Bourbon Honey Cold Brew Coffee,94.0,"San Diego, California","Valle de Cauca Department, Colombia",Light,$4.65,8.4-ounce can,NaN,False,False,False,False,$,ounce,USD,$,4.65,4.65,oz,8.4,238.14,$4.65 / 238.14g
1486,Reserve Cold Brew,94.0,"Martinez, California",Tanzania,NaN,$12.00,25.4-ounce bottle (seasonal),NaN,False,False,False,False,$,ounce,USD,$,12.00,12.00,oz,25.4,720.08,$12.0 / 720.08g
1634,Single-Origin Nitro,95.0,"Madison, Wisconsin",Ethiopia,Light,$3.49,12 ounces,NaN,False,False,False,False,$,ounce,USD,$,3.49,3.49,oz,12.0,340.19,$3.49 / 340.19g
1887,Ready-to-Drink Coffee Cold Brew,94.0,"Raleigh, North Carolina","Yirgacheffe growing region, southern Ethiopia.",NaN,$6.00,16 fluid ounces,NaN,False,False,False,False,$,fluid ounce,USD,$,6.00,6.00,oz,16.0,453.59,$6.0 / 453.59g


Retained: All rows currently selected. Dropped: None

#### price_amount & price_unit

In [62]:
# Setting to max because I want to scan the rows manually
pd.set_option('display.max_rows', None)

# Seeing if rows can be dropped due to missing critical fields
df_coffee[df_coffee[['price_unit', 'price_amount']].isna().any(axis=1)]

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,coffee_name_non_ascii,roaster_location_non_ascii,coffee_origin_non_ascii,origin_country_non_ascii,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram
124,Costa Rica Red Java Honey Finca Los Pinitos,95.0,"Charlotte, North Carolina","Alajuela, Central Valley, Costa Rica",Light,$29.00,NaN,88.0,False,False,False,False,$,NaN,USD,$,29.00,29.00,NaN,NaN,NaN,$29.0 / nang
862,Ethiopia Tamiru Tadesse Tesema Anaerobic Natural,95.0,"Yilan, Taiwan","Sidama growing region, southern Ethiopia",Medium Light,NA (available in store only),NaN,78.0,False,False,False,False,NaN,NaN,NA,NA,NaN,NaN,NaN,NaN,NaN,$nan / nang
885,Colombia Huila Finca Monteblanco Rodrigo Sanchez,94.0,"Hong Kong, China","Palestina, Huila, Colombia",Medium Light,NaN,NaN,78.0,False,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang
1309,Ecuador Eugenioides Natural Finca Perla Negra,94.0,"Minneapolis, Minnesota","Pichincha, northern Ecuador",Medium Light,NaN,NaN,73.0,False,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang
1343,Best of Panama GNEP-01 Elida Geisha Green Tip Natural,97.0,"Branford, Connecticut","Boquete growing region, western Panama",Medium Light,See website for more information,NaN,73.0,False,False,False,False,NaN,NaN,SEE,See,NaN,NaN,NaN,NaN,NaN,$nan / nang
1605,Colombia La Palma Y El Tucan Gesha,94.0,"San Diego, California","Cundinamarca Department, central Colombia",Light,NaN,NaN,84.0,False,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang
1780,Ethiopia Sidama Akrabi,94.0,"Lake, Michigan","Guji Zone, Sidama Province, southern Ethiopia.",Medium Light,$15.00,NaN,71.0,False,False,False,False,$,NaN,USD,$,15.00,15.00,NaN,NaN,NaN,$15.0 / nang
1835,Semeon Abay Ethiopia,95.0,"Lee, Massachusetts",Southern and/or western Ethiopia.,Light,NaN,NaN,84.0,False,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang
1860,Ethiopia Homacho Waeno,95.0,"Oakland, California","Aleta Wondo, Sidama (also Sidamo) growing region, southern Ethiopia.",Medium Light,NaN,NaN,75.0,False,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang
1861,Colombia Granja Esperanza Gesha AAA,95.0,"Oakland, California","Valle del Cauca growing region, Colombia.",Medium Light,NaN,NaN,76.0,False,False,False,False,NaN,NaN,NAN,nan,NaN,NaN,NaN,NaN,NaN,$nan / nang


Retained: All rows. Dropped: None. I manually looked through the short list. I could see that all important data values were present, but either the price_amount and/or price_unit data was missing, so all these rows will remain. 

### 5. Missing values

In [63]:
# Search for - Not Disclosed
df_coffee[df_coffee.astype(str).apply(lambda x: x.str.contains('Not disclosed', case=False, na=False)).any(axis=1)]

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,coffee_name_non_ascii,roaster_location_non_ascii,coffee_origin_non_ascii,origin_country_non_ascii,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram
974,Espresso Blend,94.0,"New Taipei City, Taiwan",Not disclosed,Medium,NT $450,225 grams,64.0,False,False,False,False,$,gram,TWD,NT $,450.00,14.71,g,225.0,225.00,$14.71 / 225.0g
1306,Giusto Organic Espresso Blend,94.0,"Lexington, Virginia",Not disclosed,Medium,$14.25,12 ounces,60.0,False,False,False,False,$,ounce,USD,$,14.25,14.25,oz,12.0,340.19,$14.25 / 340.19g
1602,Twisted 4.0 Espresso,94.0,"Madison, Wisconsin",Not disclosed,Medium Light,$15.75,12 ounces,75.0,False,False,False,False,$,ounce,USD,$,15.75,15.75,oz,12.0,340.19,$15.75 / 340.19g
1767,Organic Espresso,94.0,"San Diego, California",Not disclosed.,Medium,$15.00,12 ounces,60.0,False,False,False,False,$,ounce,USD,$,15.00,15.00,oz,12.0,340.19,$15.0 / 340.19g
1786,Golden Mean Espresso Blend,94.0,"Newport Beach, California",Not disclosed.,Medium,$14.95,12 ounces,60.0,False,False,False,False,$,ounce,USD,$,14.95,14.95,oz,12.0,340.19,$14.95 / 340.19g
1907,Twisted 3.0 Espresso,94.0,"Madison, Wisconsin",Not disclosed.,Medium,$15.75,12 ounces,60.0,False,False,False,False,$,ounce,USD,$,15.75,15.75,oz,12.0,340.19,$15.75 / 340.19g
1930,Ganesha Espresso,94.0,"Bellingham, Washington",Not disclosed.,Medium,$13.99,16 ounces,59.0,False,False,False,False,$,ounce,USD,$,13.99,13.99,oz,16.0,453.59,$13.99 / 453.59g
2008,Bourbon Duets,94.0,"Kaohsiung City, Taiwan",Not disclosed.,Medium,"NTD $1,000",16 ounces,59.0,False,False,False,False,$,ounce,NTD $,NTD $,1.00,NaN,oz,16.0,453.59,$nan / 453.59g
2016,Goldilocks Espresso,94.0,"Peoria, Illinois",Not disclosed.,Medium Light,$15.00,12 ounces,75.0,False,False,False,False,$,ounce,USD,$,15.00,15.00,oz,12.0,340.19,$15.0 / 340.19g
2046,Starlight Blend,96.0,"Minneapolis, Minnesota",Not disclosed.,Medium,$14.99,16 ounces,67.0,False,False,False,False,$,ounce,USD,$,14.99,14.99,oz,16.0,453.59,$14.99 / 453.59g


Looks like this phrase is only in the coffee_origin column and will convert them to NaN so keep it all clean when addressing missing values. I discovered the phrase while I was exploring and cleaning the data.

In [64]:
# Convert - Not disclosed with NaN
df_coffee['coffee_origin'] = df_coffee['coffee_origin'].replace(
    to_replace=r'(?i)^not disclosed\.?$', value=np.nan, regex=True
)

In [65]:
# Work check
# Search for - Not Disclosed
df_coffee[df_coffee.astype(str).apply(lambda x: x.str.contains('Not disclosed', case=False, na=False)).any(axis=1)]

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,coffee_name_non_ascii,roaster_location_non_ascii,coffee_origin_non_ascii,origin_country_non_ascii,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram


In [66]:
df_coffee.isna().sum().sort_values(ascending=False)

quantity_g                    149
price_unit_clean              149
price_usd                     141
price_currency                127
unit_keyword                  115
unit_quantity                 115
price_unit                    114
price_amount_clean            111
price_amount                  109
roast_level                    20
coffee_origin                  18
agtron_roast                    7
coffee_name                     0
total_score                     0
roaster_location                0
origin_country_non_ascii        0
coffee_name_non_ascii           0
roaster_location_non_ascii      0
coffee_origin_non_ascii         0
price_currency_raw              0
price_currency_clean            0
usd_per_gram                    0
dtype: int64

In [67]:
# Convert NaN in coffee_origin to Unknow
df_coffee['coffee_origin'] = df_coffee['coffee_origin'].fillna('Unknown')

In [68]:
# Work check
df_coffee[df_coffee.astype(str).apply(lambda x: x.str.contains('Unknown', case=False, na=False)).any(axis=1)]

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,coffee_name_non_ascii,roaster_location_non_ascii,coffee_origin_non_ascii,origin_country_non_ascii,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram
246,Colombia Los Nogales Yellow Bourbon,94.0,"Osaka, Japan",Unknown,Light,$28.00,150 grams,86.0,False,False,False,False,$,gram,USD,$,28.00,28.00,g,150.0,150.00,$28.0 / 150.0g
974,Espresso Blend,94.0,"New Taipei City, Taiwan",Unknown,Medium,NT $450,225 grams,64.0,False,False,False,False,$,gram,TWD,NT $,450.00,14.71,g,225.0,225.00,$14.71 / 225.0g
1306,Giusto Organic Espresso Blend,94.0,"Lexington, Virginia",Unknown,Medium,$14.25,12 ounces,60.0,False,False,False,False,$,ounce,USD,$,14.25,14.25,oz,12.0,340.19,$14.25 / 340.19g
1602,Twisted 4.0 Espresso,94.0,"Madison, Wisconsin",Unknown,Medium Light,$15.75,12 ounces,75.0,False,False,False,False,$,ounce,USD,$,15.75,15.75,oz,12.0,340.19,$15.75 / 340.19g
1767,Organic Espresso,94.0,"San Diego, California",Unknown,Medium,$15.00,12 ounces,60.0,False,False,False,False,$,ounce,USD,$,15.00,15.00,oz,12.0,340.19,$15.0 / 340.19g
1786,Golden Mean Espresso Blend,94.0,"Newport Beach, California",Unknown,Medium,$14.95,12 ounces,60.0,False,False,False,False,$,ounce,USD,$,14.95,14.95,oz,12.0,340.19,$14.95 / 340.19g
1907,Twisted 3.0 Espresso,94.0,"Madison, Wisconsin",Unknown,Medium,$15.75,12 ounces,60.0,False,False,False,False,$,ounce,USD,$,15.75,15.75,oz,12.0,340.19,$15.75 / 340.19g
1930,Ganesha Espresso,94.0,"Bellingham, Washington",Unknown,Medium,$13.99,16 ounces,59.0,False,False,False,False,$,ounce,USD,$,13.99,13.99,oz,16.0,453.59,$13.99 / 453.59g
2008,Bourbon Duets,94.0,"Kaohsiung City, Taiwan",Unknown,Medium,"NTD $1,000",16 ounces,59.0,False,False,False,False,$,ounce,NTD $,NTD $,1.00,NaN,oz,16.0,453.59,$nan / 453.59g
2016,Goldilocks Espresso,94.0,"Peoria, Illinois",Unknown,Medium Light,$15.00,12 ounces,75.0,False,False,False,False,$,ounce,USD,$,15.00,15.00,oz,12.0,340.19,$15.0 / 340.19g


I'm leaving NaN as the placeholder for missing data in numeric column types. This preserves analytical integrity by preventing skewed calculations, misleading 

#### Dropped Columns Part 2

In [72]:
df_coffee.drop(columns=[
    'coffee_name_non_ascii',
    'roaster_location_non_ascii',
    'coffee_origin_non_ascii',
    'origin_country_non_ascii'
], inplace=True)

In [73]:
print(df_coffee.columns.tolist())

['coffee_name', 'total_score', 'roaster_location', 'coffee_origin', 'roast_level', 'price_amount', 'price_unit', 'agtron_roast', 'price_currency', 'unit_keyword', 'price_currency_clean', 'price_currency_raw', 'price_amount_clean', 'price_usd', 'price_unit_clean', 'unit_quantity', 'quantity_g', 'usd_per_gram']


In [75]:
df_coffee.head()

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,price_amount,price_unit,agtron_roast,price_currency,unit_keyword,price_currency_clean,price_currency_raw,price_amount_clean,price_usd,price_unit_clean,unit_quantity,quantity_g,usd_per_gram
0,Colombia Finca Campo Hermosa,94.0,"Cleveland, Tennessee","Quindio Department, Colombia",Light,$29.99,8 ounces,82.0,$,ounce,USD,$,29.99,29.99,oz,8.0,226.8,$29.99 / 226.8g
1,Colombia Finca La Sirena Mango Co-Ferment,94.0,"Cleveland, Tennessee","Quindio Department, Colombia",Light,$22.99,8 ounces,87.0,$,ounce,USD,$,22.99,22.99,oz,8.0,226.8,$22.99 / 226.8g
2,In Bloom,94.0,"Jersey City, New Jersey",Colombia; Ethiopia,Light,$25.00,250 grams,88.0,$,gram,USD,$,25.00,25.00,g,250.0,250.0,$25.0 / 250.0g
3,Ethiopia Washed Kaffa Gimbo Lot Rich Espresso,96.0,"Chia-Yi, Taiwan","Gimbo, Kaffa Province, Ethiopia",Medium Light,NT $250,8 ounces,77.0,$,ounce,TWD,NT $,250.00,8.17,oz,8.0,226.8,$8.17 / 226.8g
4,Ethiopia Natural Gute Bona,95.0,"Chia-Yi, Taiwan","Sidamo growing region, southern Ethiopia",Medium Light,NT $400,8 ounces,78.0,$,ounce,TWD,NT $,400.00,13.07,oz,8.0,226.8,$13.07 / 226.8g


In [80]:
drop_cols = [
    'price_amount',
    'price_unit',
    'price_currency',
    'unit_keyword',
    'price_currency_clean',
    'price_currency_raw',
    'price_amount_clean',
    'unit_quantity',
    'price_unit_clean'
]

# Drop only those that are still present
df_coffee.drop(columns=[col for col in drop_cols if col in df_coffee.columns], inplace=True)

In [82]:
df_coffee.head()

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,agtron_roast,price_usd,quantity_g,usd_per_gram
0,Colombia Finca Campo Hermosa,94.0,"Cleveland, Tennessee","Quindio Department, Colombia",Light,82.0,29.99,226.8,$29.99 / 226.8g
1,Colombia Finca La Sirena Mango Co-Ferment,94.0,"Cleveland, Tennessee","Quindio Department, Colombia",Light,87.0,22.99,226.8,$22.99 / 226.8g
2,In Bloom,94.0,"Jersey City, New Jersey",Colombia; Ethiopia,Light,88.0,25.00,250.0,$25.0 / 250.0g
3,Ethiopia Washed Kaffa Gimbo Lot Rich Espresso,96.0,"Chia-Yi, Taiwan","Gimbo, Kaffa Province, Ethiopia",Medium Light,77.0,8.17,226.8,$8.17 / 226.8g
4,Ethiopia Natural Gute Bona,95.0,"Chia-Yi, Taiwan","Sidamo growing region, southern Ethiopia",Medium Light,78.0,13.07,226.8,$13.07 / 226.8g


### 6. Export

In [86]:
# Export cleaned dataset
df_coffee.to_csv(
    r"C:\Users\Chase\anaconda_projects\Exercise_6_Coffee\A6_Coffee\02_Data\Prepared_Data\Top Rated Coffee\top_rated_coffee_clean.csv",
    index=False)